In [0]:
-- =====================================================================
-- Arreglar el formato de géneros y categorías en silver.steam_games
-- ---------------------------------------------------------------------
-- El dataset de Kaggle cambió: genres/categories ahora llegan como
--   [{"id":"23","description":"Indie"}, ...]
-- y Silver los guardaba como el texto JSON completo de cada objeto.
-- Este script se queda solo con "description" (p. ej. "Indie").
-- Si un valor ya es texto plano ("Indie"), lo deja igual.
-- Es seguro ejecutarlo más de una vez.
-- =====================================================================

USE CATALOG steam_game_intelligence;

-- 0) Antes: así se ven hoy (deberías ver {"id":...,"description":...})
SELECT app_id, name, genres, categories
FROM silver.steam_games
WHERE size(genres) > 0
LIMIT 5;

-- 1) Corregir Silver en su sitio
UPDATE silver.steam_games
SET
  genres     = transform(genres,     x -> coalesce(get_json_object(x, '$.description'), x)),
  categories = transform(categories, x -> coalesce(get_json_object(x, '$.description'), x))
WHERE exists(genres,     x -> x LIKE '{%')
   OR exists(categories, x -> x LIKE '{%');

-- 2) Después: ahora deberías ver ["Indie", "Action", ...]
SELECT app_id, name, genres, categories
FROM silver.steam_games
WHERE size(genres) > 0
LIMIT 5;

-- 3) Comprobación: no debe quedar ningún valor con formato JSON (esperado: 0 y 0)
SELECT
  count_if(exists(genres,     x -> x LIKE '{%')) AS genres_con_json,
  count_if(exists(categories, x -> x LIKE '{%')) AS categories_con_json
FROM silver.steam_games;

-- 4) Géneros más frecuentes (deberían ser Indie, Action, Casual, Adventure...)
SELECT genero, count(*) AS juegos
FROM silver.steam_games
LATERAL VIEW explode(genres) AS genero
GROUP BY genero
ORDER BY juegos DESC
LIMIT 10;

-- ---------------------------------------------------------------------
-- DESPUÉS de este script:
--   Ejecuta src/gold_tables.ipynb. Reconstruye gold.dim_game (con los
--   géneros limpios), gold.game_growth (con fecha de lanzamiento) y
--   gold.twitch_steam_relationship (con el cruce de nombres completo).
--   Luego vuelve a ejecutar los dos notebooks de ML.
--
-- El notebook src/transformation/silver_games.ipynb ya incluye esta
-- misma corrección, así que el orquestador no volverá a traer el JSON.
-- ---------------------------------------------------------------------
